# Problem 9 (100 points)

Transfer learning leverages features learned on a large dataset (e.g., ImageNet with 1.2M images across 1000 classes) to solve a new task with limited data. Rather than training from scratch, we **freeze** the pretrained backbone and train only a new classification head, or **fine-tune** selected layers with a small learning rate. In this problem, you will implement the three major transfer learning strategies and build a complete pipeline.

We use the following notation in this problem.
- **Backbone**: the pretrained convolutional layers (e.g., ResNet-18 minus the final FC layer).
- **Head**: the new classification layer(s) added for the target task.
- **Freezing**: setting `param.requires_grad = False` so the optimizer ignores that parameter.
- **Feature extraction**: freeze entire backbone, train only the head.
- **Partial fine-tuning**: unfreeze later backbone layers, train with a small learning rate.
- **Full fine-tuning**: unfreeze everything, train with a very small learning rate.

In [ ]:
# Run code in this cell

"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np

torch.manual_seed(42)

> WARNING !!!
>
- Beyond importing libraries/modules/classes/functions in the preceding cell, you are **NOT allowed to import anything else for the following purposes**:
    - **As a part of your final solution.**
    - **Temporarily import something to assist you to get a solution.**

## Part 1 (10 points, non-coding task)

**Do the following tasks (Reasoning is required).**

A pretrained ResNet-18 has the following layer groups: `conv1`, `bn1`, `layer1`, `layer2`, `layer3`, `layer4`, `fc`.

1. In **feature extraction**, which layers are frozen and which are trained? If the target task has 5 classes, exactly how many parameters are trainable?
2. In **partial fine-tuning** (unfreeze `layer3`, `layer4`, and `fc`), roughly what fraction of the 11.7M total parameters are trainable? (You may estimate.)
3. Why should the learning rate for fine-tuning pretrained layers be **smaller** than the learning rate for the new head? What goes wrong if you use a large LR on pretrained weights?
4. If the target domain is very different from ImageNet (e.g., medical X-ray images), which strategy would you recommend and why?

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

Let us implement feature extraction by loading a pretrained model and replacing the head.

## Part 2 (15 points, coding task)

**Do the following tasks.**

Implement two functions:

1. `create_feature_extractor(num_classes)` that:
   - Loads a pretrained ResNet-18 via `torchvision.models.resnet18(weights='IMAGENET1K_V1')`.
   - Freezes **all** parameters (`requires_grad = False`).
   - Replaces the `fc` layer with `nn.Linear(512, num_classes)` (which has `requires_grad=True` by default).
   - Returns the model.

2. `get_trainable_params(model)` that returns a list of all parameters with `requires_grad=True`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def create_feature_extractor(num_classes):
    ...

def get_trainable_params(model):
    """Return list of parameters with requires_grad=True."""
    ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
fe_model = create_feature_extractor(5)
trainable = get_trainable_params(fe_model)
trainable_count = sum(p.numel() for p in trainable)
total_count = sum(p.numel() for p in fe_model.parameters())

assert trainable_count == 512 * 5 + 5, f"Expected 2565 trainable params, got {trainable_count}"
assert total_count > trainable_count

x = torch.randn(2, 3, 224, 224)
out = fe_model(x)
assert out.shape == (2, 5), f"Expected (2, 5), got {out.shape}"

print(f"Part 2 passed! Trainable: {trainable_count:,} / {total_count:,} ({100*trainable_count/total_count:.2f}%)")

For better performance, we can unfreeze later layers and fine-tune them with a small learning rate, using **discriminative learning rates** (different LR per layer group).

## Part 3 (15 points, coding task)

**Do the following tasks.**

1. Implement `create_partial_finetune_model(num_classes)` that:
   - Loads pretrained ResNet-18.
   - Freezes all parameters.
   - Unfreezes `layer3`, `layer4`, and the new `fc` layer.
   - Replaces `fc` with `nn.Linear(512, num_classes)`.
   - Returns the model.

2. Implement `create_optimizer_with_layer_lrs(model)` that creates an Adam optimizer with **three parameter groups**:
   - `layer3` parameters: `lr=1e-4`.
   - `layer4` parameters: `lr=1e-4`.
   - `fc` parameters: `lr=1e-3`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def create_partial_finetune_model(num_classes):
    ...

def create_optimizer_with_layer_lrs(model):
    """Create Adam optimizer with per-layer learning rates."""
    ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
pf_model = create_partial_finetune_model(5)
pf_trainable = sum(p.numel() for p in pf_model.parameters() if p.requires_grad)
pf_total = sum(p.numel() for p in pf_model.parameters())

assert pf_trainable > 2565, "More than just fc should be trainable"
assert pf_trainable < pf_total, "Not all params should be trainable"

for name, p in pf_model.named_parameters():
    if any(name.startswith(prefix) for prefix in ['conv1', 'bn1', 'layer1', 'layer2']):
        assert not p.requires_grad, f"{name} should be frozen"

optimizer = create_optimizer_with_layer_lrs(pf_model)
assert len(optimizer.param_groups) == 3, f"Expected 3 param groups, got {len(optimizer.param_groups)}"

print(f"Part 3 passed! Trainable: {pf_trainable:,} / {pf_total:,} ({100*pf_trainable/pf_total:.1f}%)")

Feature extraction can be accelerated by precomputing the backbone’s output for the entire dataset once.

## Part 4 (15 points, coding task)

**Do the following tasks.**

Implement `extract_features(model, dataloader)` that:

1. Replaces the model’s `fc` layer with `nn.Identity()`.
2. Sets the model to eval mode.
3. Iterates through the dataloader with `torch.no_grad()`, collecting all backbone outputs.
4. Returns `(features, labels)` as tensors with shapes `(N, 512)` and `(N,)`.

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
class SyntheticDataset(torch.utils.data.Dataset):
    def __init__(self, n=64, num_classes=5):
        self.images = torch.randn(n, 3, 224, 224)
        self.labels = torch.randint(0, num_classes, (n,))
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return self.images[i], self.labels[i]

syn_dataset = SyntheticDataset(64, 5)
syn_loader = torch.utils.data.DataLoader(syn_dataset, batch_size=16, shuffle=False)

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def extract_features(model, dataloader):
    """
    Extract backbone features for entire dataset.
    Returns: (features, labels) tensors of shapes (N, 512) and (N,)
    """
    ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
backbone = torchvision.models.resnet18(weights=None)
features, labels = extract_features(backbone, syn_loader)
assert features.shape == (64, 512), f"Expected (64, 512), got {features.shape}"
assert labels.shape == (64,), f"Expected (64,), got {labels.shape}"
assert not features.requires_grad, "Features should not require grad"
print(f"Part 4 passed! Features shape: {features.shape}")

Once features are extracted, training a linear classifier on top is very fast.

## Part 5 (10 points, coding task)

**Do the following tasks.**

Train a linear classifier on the extracted features from Part 4.

1. Define `classifier = nn.Linear(512, 5)`.
2. Train for 50 epochs with Adam (`lr=1e-3`) and `nn.CrossEntropyLoss`.
3. Store loss history as `loss_history` (list of 50 floats).
4. Compute final training accuracy as `train_accuracy` (float between 0 and 1).

In [ ]:
### WRITE YOUR SOLUTION HERE ###

torch.manual_seed(42)
classifier = nn.Linear(512, 5)
loss_history = []

...

train_accuracy = ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
assert len(loss_history) == 50, f"Expected 50 loss values, got {len(loss_history)}"
assert loss_history[-1] < loss_history[0], "Loss should decrease"
assert 0 <= train_accuracy <= 1
print(f"Part 5 passed! Final loss: {loss_history[-1]:.4f}, Accuracy: {train_accuracy:.2%}")

Data preprocessing must match the pretrained model’s expected input distribution.

## Part 6 (10 points, coding task)

**Do the following tasks.**

Create the standard ImageNet preprocessing pipelines.

1. `train_transform`: `RandomResizedCrop(224)` -> `RandomHorizontalFlip()` -> `ToTensor()` -> `Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])`.
2. `val_transform`: `Resize(256)` -> `CenterCrop(224)` -> `ToTensor()` -> `Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])`.
3. Implement `unnormalize(tensor)` that reverses the normalization. Input: `(C, H, W)` or `(B, C, H, W)`. Output: same shape, clipped to $[0, 1]$.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

train_transform = ...
val_transform = ...

def unnormalize(tensor):
    """Reverse ImageNet normalization. Returns tensor clipped to [0, 1]."""
    ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
original = torch.rand(3, 224, 224)
normalized = (original - mean) / std
recovered = unnormalize(normalized)
assert torch.allclose(recovered, original, atol=1e-5), "unnormalize should recover original"
assert (recovered >= 0).all() and (recovered <= 1).all()
print("Part 6 passed!")

Let us wrap everything into a unified class.

## Part 7 (15 points, coding task)

**Do the following tasks.**

Implement a `TransferLearner` class that encapsulates the full pipeline.

```python
learner = TransferLearner(num_classes=5, mode='partial_finetune')
# learner.model       -> configured model
# learner.optimizer    -> configured optimizer
# learner.trainable_count -> number of trainable parameters
# learner.total_count  -> total parameters
```

Supported modes:
- `'feature_extraction'`: freeze all backbone, train only `fc` with `lr=1e-3`.
- `'partial_finetune'`: unfreeze `layer3`, `layer4`, `fc`; backbone at `lr=1e-4`, head at `lr=1e-3`.
- `'full_finetune'`: unfreeze everything; backbone at `lr=1e-5`, head at `lr=1e-3`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class TransferLearner:
    def __init__(self, num_classes, mode='feature_extraction', backbone='resnet18'):
        ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
tl_fe = TransferLearner(5, mode='feature_extraction')
assert tl_fe.trainable_count < tl_fe.total_count * 0.01

tl_pf = TransferLearner(5, mode='partial_finetune')
assert tl_pf.trainable_count > tl_fe.trainable_count
assert tl_pf.trainable_count < tl_pf.total_count

tl_ff = TransferLearner(5, mode='full_finetune')
assert tl_ff.trainable_count == tl_ff.total_count

print(f"FE: {tl_fe.trainable_count:,} / {tl_fe.total_count:,}")
print(f"PF: {tl_pf.trainable_count:,} / {tl_pf.total_count:,}")
print(f"FF: {tl_ff.trainable_count:,} / {tl_ff.total_count:,}")
print("Part 7 passed!")

## Part 8 (10 points, non-coding task)

**Do the following tasks (Reasoning is required).**

Fill in the strategy and justification for each scenario:

| Scenario | Data size | Domain similarity | Strategy | Justification |
|---|---|---|---|---|
| Classify dog breeds (120 classes) | ~100 images/class | Very similar to ImageNet | ? | ? |
| Detect cancer in X-ray images | ~5,000 images total | Very different from ImageNet | ? | ? |
| Classify satellite images of crops | ~50,000 images | Moderately different | ? | ? |
| Classify emotions from face photos | ~500 images total | Moderately similar | ? | ? |

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """